# ViSceT5 + TWC — Finetune + Ablation · nhánh `exp/twc-ocr-centric` · Colab

Chạy **đúng quy trình của repo**: notebook chỉ đặt biến môi trường rồi gọi `bash run_all_colab.sh finetune`
(clone nhánh → `setup.sh` → `run_pipeline.py` → `training/finetune.py configs/finetune.yaml` → push checkpoint mỗi epoch lên HF).
Mọi thông số huấn luyện lấy từ **`configs/finetune.yaml`** (5 epoch, batch train/eval 32, accumulation 1, LR 1e-4 cosine, label smoothing 0.1, beam 4, chọn checkpoint theo (EM+F1)/2 trên val); notebook **không** ghi đè.
Module nào tắt thì **không được dựng** (không tham số). Loss = VQA + λ·TWC khi bật TWC (λ = `twc_answer_weight` / `twc_weight` trong yaml).
Lúc train với TWC lexical, bản OCR related là tăng cường TWA (`twc_related_train: twa`, như t1); eval/test luôn dùng bản sửa xác định.

Mỗi lần chạy = **một cấu hình**, chọn bằng các công tắc bật/tắt module:

| Công tắc | `True` | `False` |
|---|---|---|
| `USE_IMAGE` | có đặc trưng ảnh (CLIP) | **bỏ hẳn ảnh**: chỉ câu hỏi + OCR (CLIP không chạy) |
| `QAVIT` | `"question"`: QA-ViT đầy đủ — câu hỏi đưa vào 6 lớp cuối CLIP · `"no_question"`: các lớp QA-ViT có mặt nhưng câu hỏi rỗng, chỉ projection có cổng (`instruction_out_proj`, gate) học được — đúng như code trước refactor khi "tắt" QA-ViT | `"off"`: CLIP thường |
| `USE_SCESPAVIS` | OCR qua SceSpaVis (ConsFormer + char + box + det/rec) | OCR text-only |
| `USE_OCRAUG` | thêm bản OCR đã sửa (sửa xác định, **giống hệt lúc train và lúc test**) | chỉ OCR gốc |
| `OCR_RELATED` / `TWC_AUG_P` | `"augment"`: bản related = keep/correct/noise với xác suất `TWC_AUG_P`, **cùng quy trình lúc train và test** | `"correct"`: chỉ sửa xác định |
| `OCR_AUG_MODE` | `"separate"`: hai lượt ConsFormer riêng, concat đủ Origin + Related vào ViT5; TWC lấy output ViT5 | `"concat"` / `"lattice"`: các chế độ cũ để nạp checkpoint |
| `USE_TWC` | thêm loss TWC khi train (eval/test không đổi) | không TWC |
| `TWC_OBJECTIVE` | `answer`: câu hỏi phải tìm đúng các từ OCR (gốc hoặc bản sửa) chứa đáp án giữa mọi từ OCR trong batch · `lexical`: từ OCR gốc ↔ bản related (TWA) · `both` | — |
| `TWC_QUEUE` | (chỉ `lexical`/`both`, loss infonce) số đặc trưng từ của các batch trước làm negative thêm | `0`: chỉ so trong batch |
| `OCR_NORM` | `layernorm`: chuẩn hoá từng luồng OCR như code trước refactor (run Vast 31.7) | `rmsnorm`: bản refactor |

TWC `answer` dùng softmax InfoNCE nhiều positive trên đầu ra encoder: truy vấn = trung bình trạng thái encoder của câu hỏi, ứng viên = mọi từ OCR trong batch, positive = từ OCR chỉ gồm các từ của đáp án (~81% mẫu train có), từ trùng chữ đáp án ở ảnh khác bị bỏ qua. TWC `lexical` dùng đặc trưng chữ (ConsFormer + ký tự) và bộ nhớ cross-batch.

**Vì sao TWC `answer`:** TWA gốc căn chỉnh OCR gốc ↔ OCR related vì đầu ra của TWA là bộ dự đoán chọn từ trong vocab/OCR, nên biểu diễn từ OCR phải khớp trực tiếp với từ đúng. ViSceT5 sinh đáp án bằng decoder, nên ở đây giữ ý tưởng tương phản mức từ của TWC nhưng neo vào đáp án: câu hỏi phải tìm đúng các từ OCR (gốc hoặc bản sửa) chứa đáp án. Chế độ `lexical` (TWA nguyên bản) và `both` giữ lại để làm ablation.

**Các lựa chọn khác (đã thử):** batch 8 × 2 (các run hiệu dụng 16 đều chỉ ~29.7 test EM), `OCR_NORM="layernorm"`, `TWC_OBJECTIVE="answer"` (trọng số 0.1), bản related lúc train = bản sửa xác định như lúc test.

**Các run t1 cũ là tham chiếu lịch sử; cấu hình thực thi hiện tại nằm trong cell bên dưới và YAML.**

**Luồng đầy đủ khi bật augmentation, ConsFormer và TWC:** không chọn token lattice, không type embedding. Mỗi bản giữ character riêng và bbox/det/rec tương ứng. ViT5 encoder nhận question + image + toàn bộ Origin + toàn bộ Related. VQA sinh câu trả lời bình thường; TWC đối chiếu feature Origin/Related từ cùng output ViT5 encoder (fused), sau khi pool subword về entry. `OCR_RELATED="augment"`, p=0.35: keep/correct/noise, cùng quy trình và seed theo mẫu ở train/test.

**ViSceT5 + TWC:** `USE_IMAGE=True, QAVIT="off"/"no_question", USE_SCESPAVIS=True, USE_OCRAUG=True, USE_TWC=True`. TWC `lexical`/`both` cần `USE_OCRAUG=True`.

CLIP luôn đóng băng; QA-ViT chỉ thêm các module học được vào 6 lớp cuối (`instruct_dim_reduce` chiếu câu hỏi, `instruction_out_proj` khởi tạo = `out_proj`, gate khởi tạo 0).
**Ablation hiện tại:** bật ConsFormer/GroupAttention đã sửa, tắt OCR augmentation và TWC. Giữ image, question và OCR Origin + character/bbox/det/rec. Run mới có hậu tố `groupfix` để phân biệt checkpoint của công thức cũ.


## 1. Lấy repo (nhánh `exp/twc-ocr-centric`)

In [ ]:
import os
BRANCH = "exp/twc-ocr-centric"
if not os.path.exists("/content/ViSceT5"):
    !git clone -b {BRANCH} https://github.com/Kussssssss/ViSceT5.git /content/ViSceT5
%cd /content/ViSceT5
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!git log --oneline -3
assert os.path.exists("data/ocr_correction.py") and "def _twc_memory" in open("models/openvivqa_model.py", encoding="utf-8").read(), "Sai nhánh hoặc code cũ — cần exp/twc-ocr-centric có bộ nhớ TWC!"

## 2. Hugging Face token (để `finetune.py` tự push checkpoint mỗi epoch, tự resume khi Colab ngắt, và upload cuối)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 3. Cấu hình — bật/tắt module

In [ ]:
USE_IMAGE     = True
QAVIT         = "no_question"
USE_SCESPAVIS = True
USE_CONSFORMER = True
USE_OCRAUG    = False
OCR_AUG_MODE  = "separate"
OCR_RELATED   = "augment"
TWC_AUG_P     = 0.35
USE_TWC       = False
TWC_OBJECTIVE = "lexical"
TWC_QUEUE     = 0
TWC_TAU       = 0.07
OCR_NORM      = "layernorm"
DATASET       = "ViTextVQA"
PREDICT_CKPT  = "best"

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

assert QAVIT in ("off", "no_question", "question"), "QAVIT: off | no_question | question"
assert OCR_AUG_MODE in ("separate", "lattice", "concat"), "OCR_AUG_MODE: separate | lattice | concat"
assert OCR_RELATED in ("augment", "correct") and 0 <= TWC_AUG_P <= 1
assert USE_IMAGE or QAVIT == "off", "QA-ViT cần ảnh: khi USE_IMAGE=False phải đặt QAVIT=\"off\""
assert TWC_OBJECTIVE in ("answer", "lexical", "both") and OCR_NORM in ("layernorm", "rmsnorm")
assert USE_OCRAUG or not (USE_TWC and TWC_OBJECTIVE != "answer"), "TWC lexical cần nửa OCR related: bật USE_OCRAUG"
_twc_tag = "twc-" + {"answer": "ans", "lexical": f"lex-q{TWC_QUEUE}", "both": f"both-q{TWC_QUEUE}"}[TWC_OBJECTIVE]
RUN_NAME = "-".join(m for m, on in (("noimg", not USE_IMAGE), ("qavit", QAVIT == "question"), ("scespavis", USE_SCESPAVIS),
                                    ("ocraug", USE_OCRAUG), (OCR_AUG_MODE + (f"-p{int(round(100 * TWC_AUG_P))}" if OCR_RELATED == "augment" else ""), USE_OCRAUG), (_twc_tag, USE_TWC), ("rms", OCR_NORM == "rmsnorm"), ("noadapter", USE_IMAGE and QAVIT == "off")) if on) or "base"
if not USE_OCRAUG and not USE_TWC:
    RUN_NAME += "-noaug-notwc"
if USE_SCESPAVIS and not USE_CONSFORMER:
    RUN_NAME += "-noconsformer"
if USE_SCESPAVIS and USE_CONSFORMER:
    RUN_NAME += "-groupfix"
OUTPUT_DIR = f"/content/ViSceT5/output/twc_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-twcbr-{DATASET.lower()}-{RUN_NAME}"
_b = lambda v: "true" if v else "false"

os.environ.update({
    "GIT_BRANCH":           "exp/twc-ocr-centric",
    "ABLATION_USE_IMAGE":   _b(USE_IMAGE),
    "QAVIT_MODE":           QAVIT if USE_IMAGE else "off",
    "ABLATION_USE_OCR":     _b(USE_SCESPAVIS),
    "ABLATION_USE_CONSFORMER": _b(USE_CONSFORMER),
    "ABLATION_USE_OCR_AUG": _b(USE_OCRAUG),
    "OCR_AUG_MODE":         OCR_AUG_MODE,
    "OCR_RELATED":          OCR_RELATED,
    "TWC_ADV_PROBABILITY":  str(TWC_AUG_P),
    "USE_TWC":              _b(USE_TWC),
    "TWC_FEATURE":          "fused",
    "TWC_VIEW":             "shared",
    "TWC_OBJECTIVE":        TWC_OBJECTIVE,
    "TWC_QUEUE_SIZE":       str(int(TWC_QUEUE)),
    "TWC_TEMPERATURE":      str(TWC_TAU),
    "OCR_FEATURE_NORM":     OCR_NORM,
    "DATASET_NAME":         DATASET,
    "OUTPUT_DIR":           OUTPUT_DIR,
    "HF_REPO":              HF_REPO,
})
_main = USE_IMAGE and QAVIT != "question" and USE_SCESPAVIS and USE_CONSFORMER and USE_OCRAUG
print(f"Cấu hình: Ảnh={USE_IMAGE} | QA-ViT={QAVIT} | SceSpaVis={USE_SCESPAVIS} | ConsFormer={USE_CONSFORMER} | OCR-Aug={USE_OCRAUG}{' (' + OCR_AUG_MODE + ', related=' + OCR_RELATED + (f' p={TWC_AUG_P}' if OCR_RELATED == 'augment' else '') + ')' if USE_OCRAUG else ''} | TWC={USE_TWC}"
      f"{' (' + TWC_OBJECTIVE + ')' if USE_TWC else ''} | OCR norm={OCR_NORM} | τ cố định={TWC_TAU}"
      f"{'   ← ViSceT5' + (' + TWC' if USE_TWC else '') if _main else ''}")
print(f"RUN_NAME={RUN_NAME}\nOUTPUT_DIR={OUTPUT_DIR}\nHF_REPO={HF_REPO}")
print(f"PREDICT_CKPT={PREDICT_CKPT}")
print("Thông số huấn luyện: lấy từ configs/finetune.yaml")

## 4. (Tuỳ chọn) Mock test — vài bước để bắt lỗi trước khi chạy thật
Dùng chế độ `mock` của `run_all_colab.sh` (`--smoke_test`: 8 mẫu train, 4 mẫu val, 3 bước); không push, không resume từ HF. `TWC_QUEUE_START=1` để bộ nhớ TWC lexical (nếu dùng) được đọc ngay từ bước 2. Với TWC `answer`, log phải có `twc_answer_loss`, `twc_answer_acc`.

In [ ]:
!HF_REPO= OUTPUT_DIR={OUTPUT_DIR}_mock TWC_QUEUE_START=1 bash run_all_colab.sh finetune mock

## 4b. (Tuỳ chọn) Kiểm tra tương đương với code cũ — không cần train
Lấy checkpoint đã train bằng code cũ (`Kus669/ViSceT5-ft-vitextvqa-only_consformer`, `checkpoint-21975` — chính checkpoint sinh ra 31.72), chuyển sang định dạng nhánh này (`scripts/port_old_checkpoint.py`: cùng trọng số; LayerNorm OCR + adapter CLIP như code cũ; hàng embedding ký tự ánh xạ lại theo từng ký tự), rồi predict bằng **toàn bộ pipeline của nhánh này** (đọc dữ liệu, lọc OCR, xử lý ảnh, forward, beam search) và chấm theo benchmark.

- Test EM ≈ 31.7 và trùng ~95–100% câu trả lời với file nộp gốc → phần xử lý + inference của nhánh này tương đương code cũ; chênh lệch khi train lại chỉ còn do quá trình huấn luyện.
- Thấp hơn rõ / trùng ít → còn khác biệt trong pipeline (dữ liệu, OCR, forward) cần tìm tiếp.

Kết quả được upload vào repo HF `…-parity-only_consformer`.


In [ ]:
import os, shutil
PARITY_REPO = f"{HF_USER}/ViSceT5-twcbr-parity-only_consformer"
PORT_DIR = "/content/ported_only_consformer"
PARITY_DIR = "/content/parity_only_consformer"
os.makedirs(PARITY_DIR, exist_ok=True)
!bash setup.sh
if not os.path.exists(f"{PORT_DIR}/model.safetensors"):
    !python scripts/port_old_checkpoint.py --repo Kus669/ViSceT5-ft-vitextvqa-only_consformer --ckpt checkpoint-21975 --out {PORT_DIR}
if not os.path.exists(f"{PARITY_DIR}/submission_test.csv"):
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{sp}.csv")
    !STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET=ViTextVQA PREDICT_CKPT_DIR={PORT_DIR} HF_REPO= bash run_all_colab.sh
    for sp in ("dev", "test"):
        shutil.copy2(f"/content/ViSceT5/output/submission_{sp}.csv", f"{PARITY_DIR}/submission_{sp}.csv")
    shutil.copy2("/content/ViSceT5/train_execution.log", f"{PARITY_DIR}/predict_log.txt")
for line in open(f"{PARITY_DIR}/predict_log.txt", encoding="utf-8", errors="ignore"):
    if "[dev] EM=" in line:
        print(line.strip())
!pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap
!python -m training.benchmark_eval --gt eval_gt/ViTextVQA_test_gt.json --pred {PARITY_DIR}/submission_test.csv --out {PARITY_DIR}/eval_test.json
!python scripts/repro_compare.py --gt eval_gt/ViTextVQA_test_gt.json --ref repro/vast_only_consformer_test.csv --pred vast_original=repro/vast_only_consformer_test.csv this_branch_inference={PARITY_DIR}/submission_test.csv --out {PARITY_DIR}/parity_compare.json
if os.environ.get("HF_TOKEN"):
    !python scripts/hf_io.py upload_predictions --repo {PARITY_REPO} --dir {PARITY_DIR} --path_in_repo parity --files submission_dev.csv submission_test.csv predict_log.txt eval_test.json parity_compare.json


## 5. Finetune (từ đầu, không pretrain) — log train có thêm `twc_answer_loss`/`twc_answer_acc` (answer) hoặc `twc_loss`/`twc_acc` (lexical) khi bật TWC

In [ ]:
!bash run_all_colab.sh finetune

## 5b. (Khi cần) Đẩy checkpoint local lên HF rồi train tiếp
Dùng khi đã train mà **chưa có `HF_TOKEN`** (checkpoint chỉ nằm trên Colab): chạy cell 2 (token) → cell 3 (cấu hình, đúng công tắc của lần train) → cell này. Cell đẩy các `checkpoint-N` trong `OUTPUT_DIR` (kèm optimizer/scheduler để resume) lên `HF_REPO`, sau đó chạy lại **cell 5**: `finetune.py` tự resume từ checkpoint mới nhất trên HF và từ đó push mỗi epoch. Chỉ checkpoint đã lưu cuối epoch mới có; phần đang train dở của epoch sau bị mất. Nếu runtime Colab đã reset thì không còn gì để đẩy.


In [ ]:
!ls -d {OUTPUT_DIR}/checkpoint-* 2>/dev/null
!python scripts/hf_io.py upload_checkpoints --repo {HF_REPO} --dir {OUTPUT_DIR}


## 6. Dự đoán dev + test (`STAGE=predict` của `run_pipeline.py`) và chấm điểm
`predict.py` in EM/F1 trên dev và ghi file nộp cho test. File được chép vào `OUTPUT_DIR` để không đè giữa các cấu hình.
Test được chấm bằng `training/benchmark_eval.py` — **đúng giao thức benchmark** (cùng cách chấm các dòng baseline/LMM trong bài): EM, F1 token/char, BLEU-1..4/avg, CIDEr, BERTScore (mBERT).
Đáp án test lấy từ `eval_gt/<DATASET>_test_gt.json` trong repo (đã có `ViTextVQA`); dataset chưa có file thì cell hiện nút upload **ngay đầu cell** (trước khi predict).
Nếu `OUTPUT_DIR` (hoặc `predictions/` trên HF) đã có `submission_test.csv` thì cell **bỏ qua predict** và chỉ chấm điểm.
File nộp, log predict (có EM/F1 dev) và kết quả chấm test được upload vào thư mục `predictions/` của repo HF của cấu hình.

**Chọn checkpoint:** `PREDICT_CKPT` ở cell 3. Cell in bảng các checkpoint kèm EM/F1/CIDEr trên val của từng epoch rồi dùng checkpoint được chọn:
`best` = model tốt nhất (thư mục gốc, do `trainer.save_model` lưu sau `load_best_model_at_end`), `latest`, `epoch3`, hoặc `checkpoint-17580`.
Chỉ tải file inference của checkpoint đó (không tải optimizer). Kết quả của `best` nằm ở `OUTPUT_DIR` và `predictions/` trên HF;
của checkpoint khác nằm ở `OUTPUT_DIR/predict_checkpoint-N` và `predictions/checkpoint-N/` → không đè nhau.

**Predict lại sau khi Colab đã tắt:** chạy cell 1, 2, 3 (đặt đúng công tắc của lần train và `PREDICT_CKPT`) rồi chạy thẳng cell này, bỏ qua mock và finetune.

In [ ]:
import os, shutil, glob
!git pull -q origin {BRANCH}
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    TEST_GT_JSON = f"/content/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    from google.colab import files
    print(f"⬆️ Chọn file {os.path.basename(TEST_GT_JSON)} để upload (bấm Cancel nếu chưa có → vẫn predict, chưa chấm test)")
    up = files.upload()
    if up:
        shutil.move(os.path.abspath(next(iter(up))), TEST_GT_JSON)
print("GT test:", TEST_GT_JSON if os.path.exists(TEST_GT_JSON) else "chưa có")

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, PRED_DIR, PRED_REPO_PATH = CKPT_INFO["model_dir"], CKPT_INFO["pred_dir"], CKPT_INFO["pred_path_in_repo"]
assert os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")), f"Không có model trong {MODEL_DIR}"

tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
pred = os.path.join(PRED_DIR, f"submission_{tag}test.csv")
if os.path.exists(pred):
    print(f"✅ Đã có {pred} → bỏ qua predict, chỉ chấm điểm.")
else:
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{tag}{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{tag}{sp}.csv")
    !STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={DATASET} PREDICT_CKPT_DIR={MODEL_DIR} HF_REPO= bash run_all_colab.sh
    for sp in ("dev", "test"):
        src = f"/content/ViSceT5/output/submission_{tag}{sp}.csv"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(PRED_DIR, os.path.basename(src)))
            print("→", os.path.join(PRED_DIR, os.path.basename(src)))
    shutil.copy2("/content/ViSceT5/train_execution.log", os.path.join(PRED_DIR, "predict_log.txt"))
log = os.path.join(PRED_DIR, "predict_log.txt")
dev_lines = [l.strip() for l in open(log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l] if os.path.exists(log) else []
if dev_lines:
    for line in dev_lines[-2:]:
        print(line)
if os.path.exists(TEST_GT_JSON) and os.path.exists(pred):
    !pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap
    !python -m training.benchmark_eval --gt {TEST_GT_JSON} --pred {pred} --out {PRED_DIR}/eval_test.json
else:
    print("ℹ️ Chưa đặt TEST_GT_JSON → chưa chấm test.")

if os.environ.get("HF_TOKEN") and HF_REPO:
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {PRED_DIR} --path_in_repo {PRED_REPO_PATH} --files submission_{tag}dev.csv submission_{tag}test.csv predict_log.txt eval_test.json
else:
    print("⚠️ Không có HF_TOKEN/HF_REPO → file dự đoán chỉ nằm trên Colab, hãy tải về trước khi ngắt phiên.")

## 7. Tổng hợp kết quả mọi cấu hình đã chạy

In [ ]:
import json, os, glob
import pandas as pd
rows = []
for od in sorted(glob.glob(f"/content/ViSceT5/output/twc_{DATASET}_*")):
    name = od.split(f"twc_{DATASET}_")[-1]
    mods = set(name.split("-"))
    r = {"run": name, "Ảnh": "noimg" not in mods, "QA-ViT": "qavit" in mods,
         "SceSpaVis": "scespavis" in mods, "OCR-Aug": "ocraug" in mods, "TWC": "twc" in mods}
    st = sorted(glob.glob(os.path.join(od, "checkpoint-*", "trainer_state.json")),
                key=lambda p: int(p.split("checkpoint-")[-1].split(os.sep)[0]))
    if st:
        hist = json.load(open(st[-1])).get("log_history", [])
        key = "eval_cider" if DATASET == "OpenViVQA" else "eval_em_f1"
        evs = [h for h in hist if "eval_em" in h]
        if evs:
            best = max(evs, key=lambda h: h.get(key, 0.5 * (h.get("eval_em", 0) + h.get("eval_f1", 0))))
            r.update({"val_EM": round(100 * best.get("eval_em", float("nan")), 2), "val_F1": round(100 * best.get("eval_f1", 0), 2),
                      "val_CIDEr": round(best.get("eval_cider", float("nan")), 4), "best_epoch": best.get("epoch")})
    for pd_dir in [od] + sorted(glob.glob(os.path.join(od, "predict_checkpoint-*"))):
        rr = dict(r, ckpt="best" if pd_dir == od else os.path.basename(pd_dir)[len("predict_"):])
        ev = os.path.join(pd_dir, "eval_test.json")
        if os.path.exists(ev):
            t = json.load(open(ev, encoding="utf-8"))
            rr.update({"test_EM": round(100 * t.get("Exact Match (EM)", float("nan")), 2), "test_F1": round(100 * t.get("F1 (Token-level)", float("nan")), 2),
                       "test_BLEU4": round(t.get("BLEU-4", float("nan")), 4), "test_CIDEr": round(t.get("CIDEr", float("nan")), 4),
                       "test_BERTScore": round(100 * t.get("BERTScore_F1", float("nan")), 2)})
        elif pd_dir != od:
            continue
        rows.append(rr)
df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "Chưa có cấu hình nào chạy xong.")

## 8. Trực quan hoá attention (cấu hình hiện tại)
`scripts/visualize_attention.py` sinh đáp án (beam search như predict) rồi lấy **cross-attention** của các token đáp án lên đầu vào encoder,
trung bình trên mọi lớp và mọi head. Mỗi câu hỏi ra một hình: heatmap trên 14×14 patch ảnh (nếu mô hình dùng ảnh), box OCR tô màu theo attention,
và attention trên từng từ của câu hỏi, OCR gốc, OCR đã sửa (viền đỏ = từ bị sửa). `segment_mass.csv` ghi tỷ lệ attention dồn vào từng phần.
Dùng checkpoint `PREDICT_CKPT` (cell 3). Đặt `ATTN_IDS` để chọn câu hỏi cụ thể; để trống thì lấy ngẫu nhiên `ATTN_N` câu (cùng seed cho mọi cấu hình). Hình được upload vào `attention/` (hoặc `attention/checkpoint-N/`) trên repo HF.

In [ ]:
import os, glob
from IPython.display import Image, display
ATTN_SPLIT = "test"
ATTN_N = 8
ATTN_IDS = []

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, ATTN_DIR = CKPT_INFO["model_dir"], os.path.join(CKPT_INFO["pred_dir"], "attention")
ATTN_REPO_PATH = CKPT_INFO["attn_path_in_repo"]
ids_arg = ("--ids " + " ".join(map(str, ATTN_IDS))) if ATTN_IDS else f"--n {ATTN_N}"
gt_arg = f"--gt eval_gt/{DATASET}_test_gt.json" if os.path.exists(f"eval_gt/{DATASET}_test_gt.json") else ""
!python scripts/visualize_attention.py --ckpt_dir {MODEL_DIR} --dataset {DATASET} --split {ATTN_SPLIT} {ids_arg} {gt_arg} --out_dir {ATTN_DIR}

pngs = sorted(glob.glob(os.path.join(ATTN_DIR, "*.png")))
for p in pngs:
    display(Image(filename=p))
if os.environ.get("HF_TOKEN") and HF_REPO and pngs:
    names = " ".join(os.path.basename(p) for p in pngs + [os.path.join(ATTN_DIR, "segment_mass.csv")])
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {ATTN_DIR} --path_in_repo {ATTN_REPO_PATH} --files {names}